# Day 5 — 시각화 원칙: 정직한 그림 3장 그리기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-1/day-05.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-1/day-05.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 평균은 같은데 전혀 다른 두 집단

In [ ]:
import numpy as np

a = np.array([4.8, 4.9, 5.0, 5.1, 5.2])
b = np.array([1.0, 1.2, 5.0, 8.8, 9.0])
print("A 평균:", round(a.mean(), 2), " B 평균:", round(b.mean(), 2))
print("A 범위:", a.min(), "~", a.max(), " B 범위:", b.min(), "~", b.max())

### 예시 2 — 연습용 가상 데이터 만들기 + histogram

PD군과 대조군 각 30명의 나이와 "보행 시간 변동성"(걸음마다 걸리는 시간이 얼마나 들쭉날쭉한가)을 가상으로 만듭니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("../figures", exist_ok=True)   # figures 폴더가 없으면 만든다

rng = np.random.default_rng(42)  # 같은 숫자(seed)를 쓰면 매번 같은 가상 데이터가 나온다

n_control, n_pd = 30, 30
df = pd.DataFrame({
    "group": ["Control"] * n_control + ["PD"] * n_pd,
    "age": np.concatenate([rng.normal(62, 8, n_control), rng.normal(66, 8, n_pd)]).round(0),
})
# 보행 변동성(%): 나이가 많을수록, PD군일수록 조금 크게 만든 가상 값
df["stride_cv"] = (
    1.0 + 0.03 * (df["age"] - 60) + np.where(df["group"] == "PD", 0.8, 0.0)
    + rng.normal(0, 0.4, len(df))
).round(2)

print(df.head())
print(df.groupby("group")["stride_cv"].describe().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["stride_cv"], bins=12, color="gray", edgecolor="white")
ax.set_xlabel("Stride time variability (CV, %)")
ax.set_ylabel("Number of participants")
ax.set_title("Distribution of stride variability (simulated, n=60)")
fig.tight_layout()
fig.savefig("../figures/fig01_histogram.png", dpi=150)
plt.show()

### 예시 3 — boxplot + 개별 점, 그리고 scatter plot

예시 2와 **같은 노트북에서 이어서** 실행합니다(`df`와 `rng`를 그대로 씁니다).

In [ ]:
colors = {"Control": "#0072B2", "PD": "#D55E00"}  # 색각 이상에도 구분되는 파랑/주황
groups = ["Control", "PD"]

# --- boxplot + 개별 점 ---
fig, ax = plt.subplots(figsize=(5, 4))
data = [df.loc[df["group"] == g, "stride_cv"] for g in groups]
ax.boxplot(data, showfliers=False)          # 이상치 점은 아래에서 직접 그리므로 끈다
ax.set_xticks([1, 2])
ax.set_xticklabels(groups)
for i, g in enumerate(groups, start=1):
    y = df.loc[df["group"] == g, "stride_cv"]
    x = rng.normal(i, 0.05, len(y))          # 점이 겹치지 않게 살짝 흩뿌린다(jitter)
    ax.scatter(x, y, color=colors[g], alpha=0.6, s=20)
ax.set_xlabel("Group")
ax.set_ylabel("Stride time variability (CV, %)")
ax.set_title("Stride variability by group (simulated)")
fig.tight_layout()
fig.savefig("../figures/fig02_boxplot.png", dpi=150)
plt.show()

# --- scatter plot ---
fig, ax = plt.subplots(figsize=(6, 4))
for g in groups:
    sub = df[df["group"] == g]
    ax.scatter(sub["age"], sub["stride_cv"], color=colors[g], label=g, alpha=0.7)
ax.set_xlabel("Age (years)")
ax.set_ylabel("Stride time variability (CV, %)")
ax.set_title("Age vs. stride variability (simulated)")
ax.legend(title="Group")
fig.tight_layout()
fig.savefig("../figures/fig03_scatter.png", dpi=150)
plt.show()

## 직접 해보기

2. **예시 1, 2, 3을 순서대로 입력해 실행합니다.** 그림 세 장이 노트북에 보이고, `figures/` 폴더에 PNG 파일 세 개가 생겼는지 확인합니다.

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# ls ~/Documents/neurodegeneration-bioinformatics/figures

4. **실험 2 — 군별로 겹친 histogram.** 아래 코드를 실행해, 예시 2의 "넓적한 분포"가 사실은 두 봉우리였음을 확인합니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for g in groups:
    ax.hist(df.loc[df["group"] == g, "stride_cv"], bins=10, alpha=0.6, color=colors[g], label=g)
ax.set_xlabel("Stride time variability (CV, %)")
ax.set_ylabel("Number of participants")
ax.set_title("Stride variability by group (simulated)")
ax.legend(title="Group")
fig.tight_layout()
plt.show()

8. **저장, 일지, commit.**

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# cd ~/Documents/neurodegeneration-bioinformatics
# git add notebooks/01_setup_and_metadata.ipynb figures/ learning-log.md
# git commit -m "Day 5: histogram, boxplot, scatter plot 작성"